In [28]:
# Cell 1 — Imports and path setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import numpy as np
import pandas as pd
import mlflow
from src.common.utils.config import load_config
from src.modeling.data_loader     import load_cell
from src.modeling.feature_selector import get_feature_columns
from src.modeling.splitter         import time_split
from src.modeling.metrics          import mape, wmape
from src.modeling.regressor        import train_regressor
from src.modeling.classifier       import train_classifier
from src.modeling.two_stage        import train_two_stage
from src.modeling.model_io         import save_model

In [29]:
# Cell 2 — Configurable variables (ONLY cell that changes per notebook)
# === CONFIGURABLE VARIABLES — edit these for each model cell ===
CLUSTER_NAME = "Low-Value Sporadic"  # "Power" | "High-Value Active" | "Low-Value Sporadic"
DEMAND_CLASS = "Intermittent"        # "Continuous" | "Intermittent" | "Lumpy"
GRID_CELL    = 7                     # see mapping table below

# Grid cell reference:
# cluster \ demand   Continuous  Intermittent  Lumpy
# Power                  0            1           2
# High-Value Active      3            4           5   <- two-stage
# Low-Value Sporadic     6            7           8   <- two-stage
MODEL_NAME = f"model_{GRID_CELL}_{CLUSTER_NAME.lower().replace(' ', '_')}_{DEMAND_CLASS.lower()}"
IS_TWO_STAGE = GRID_CELL in {5, 8}
print(f"MODEL_NAME: {MODEL_NAME} | IS_TWO_STAGE: {IS_TWO_STAGE}")

MODEL_NAME: model_7_low-value_sporadic_intermittent | IS_TWO_STAGE: False


In [30]:
# Cell 3 — Load config and data
cfg = load_config()
mod_cfg = cfg["modeling"]


cell_cache_path  = f"../../data/intermediate/cell_{GRID_CELL}_{CLUSTER_NAME.lower().replace(' ', '_')}_{DEMAND_CLASS.lower()}.parquet"
matrix_path = "../../data/intermediate/"  + "full_reduced_modeling_matrix.parquet"
df = load_cell(matrix_path, CLUSTER_NAME, DEMAND_CLASS)
df.to_parquet(cell_cache_path, index=False)
print(f"Filtered and cached to: {cell_cache_path}")

df = pd.read_parquet(cell_cache_path)


print(f"Cell rows: {len(df):,} | Columns: {df.shape[1]}")

Filtered and cached to: ../../data/intermediate/cell_7_low-value_sporadic_intermittent.parquet
Cell rows: 4,940,437 | Columns: 57


In [31]:
# Cell 4 — Split
df_train, df_eval = time_split(df, mod_cfg["train_end"], mod_cfg["eval_start"])
print(f"Train: {len(df_train):,} rows ({mod_cfg['train_start']} – {mod_cfg['train_end']})")
print(f"Eval:  {len(df_eval):,} rows ({mod_cfg['eval_start']} – {mod_cfg['eval_end']})")

Train: 4,132,477 rows (2015-04 – 2018-12)
Eval:  807,960 rows (2019-01 – 2019-10)


In [32]:
# Cell 5 — Feature selection
# For two-stage models, feature_cols are shared across both stages
target_col    = mod_cfg["target_regressor"]
feature_cols, target_col = get_feature_columns(df_train, target_col, mod_cfg["drop_columns"])
print(f"Feature count: {len(feature_cols)}")
print(feature_cols)

Feature count: 41
['net_sales', 'return_quantity', 'return_sales', 'n_invoices', 'n_returns', 'year', 'month_int', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3', 'is_peak_month', 'is_trough_month', 'lag_sales_1m', 'lag_sales_2m', 'lag_sales_3m', 'lag_sales_6m', 'lag_sales_12m', 'roll_mean_qty_3m', 'roll_std_qty_12m', 'roll_mean_sales_3m', 'roll_mean_sales_12m', 'roll_nonzero_count_12m', 'roll_cv_qty_3m', 'roll_cv_qty_12m', 'yoy_growth', 'short_long_ratio', 'trend_slope_approx', 'roll_return_rate_3m', 'roll_return_rate_6m', 'price_per_unit', 'months_since_last_purchase', 'demand_event_rate_6m', 'demand_class_encoded', 'sku_activity_rate', 'is_structural_collapse', 'is_hyper_growth', 'sku_log_total_qty', 'seasonal_index']


In [33]:
# Cell 6 — Train
mlflow.set_tracking_uri(mod_cfg["mlflow_tracking_uri"])
mlflow.set_experiment(mod_cfg["mlflow_experiment"])

if IS_TWO_STAGE:
    clf, reg = train_two_stage(df_train, df_eval, feature_cols, cfg, MODEL_NAME)
else:
    reg = train_regressor(
        df_train, df_eval, feature_cols, target_col,
        mod_cfg["xgb_param_grid"], cfg, MODEL_NAME
    )
    clf = None

[model_7_low-value_sporadic_intermittent] train: 4132477 rows → 4132477 kept (0 removed)
[model_7_low-value_sporadic_intermittent] eval:  807960 rows → 807960 kept (0 removed)

  model_7_low-value_sporadic_intermittent_search  (1 candidates, 3-fold TimeSeriesCV)
  scoring: neg_mean_absolute_error
  [1/1]  score=-3.0189 ± 0.4607  <-- best
          subsample: 1.0
          reg_lambda: 0.5
          reg_alpha: 0
          n_estimators: 200
          min_child_weight: 5
          max_depth: 7
          learning_rate: 0.1
          colsample_bytree: 1.0

  Best CV score : -3.0189
  Best params   : {'subsample': 1.0, 'reg_lambda': 0.5, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 1.0}



/Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/src/modeling/regressor.py:73: RuntimeWarning: overflow encountered in expm1
  y_pred_raw = np.expm1(y_pred_log)
/Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/src/modeling/regressor.py:82: RuntimeWarning: overflow encountered in expm1
  y_train_pred_raw = np.expm1(y_train_pred_log)


In [34]:
# Cell 7 — Evaluate
from src.modeling.two_stage import predict_two_stage

X_eval = df_eval[feature_cols].values
y_true_raw = df_eval["target_qty_raw"].values

if IS_TWO_STAGE:
    y_pred_raw = predict_two_stage(clf, reg, X_eval, mod_cfg["classifier_threshold"])
else:
    y_pred_raw = reg.predict(X_eval)

mape_val, zero_frac = mape(y_true_raw, y_pred_raw)
wmape_val = wmape(y_true_raw, y_pred_raw)
print(f"MAPE  : {mape_val:.2f}%  (zero actuals excluded: {zero_frac:.1%})")
print(f"WMAPE : {wmape_val:.2f}%")

MAPE  : 88.26%  (zero actuals excluded: 96.7%)
WMAPE : 316.30%


In [35]:
# Cell 8 — Save models
output_dir = Path(mod_cfg["model_output_path"])
if IS_TWO_STAGE:
    save_model(clf, output_dir, f"{MODEL_NAME}_stage1_clf")
    save_model(reg, output_dir, f"{MODEL_NAME}_stage2_reg")
else:
    save_model(reg, output_dir, MODEL_NAME)
print(f"Model(s) saved to {output_dir}")

Model(s) saved to ../../../data/output/models
